**Auteur(s)** : Cheikhou Akhmed KANE

**Description** : Mise en place `parcelles.shp`

# Création du Fichier des Parcelles

## 1. Description du projet

Ce notebook a pour objectif de générer le fichier `parcelles.shp`, une des couches d'information géographique requise par le module agricole de MAELIA. Cette couche est une subdivision des îlots qui permet d'assigner différentes séquences de culture. Dans notre cas, chaque îlot correspondra à une seule parcelle.

---
## 2. Objectifs

* Charger le fichier `ilots.shp` comme base géométrique et attributaire.
* Créer un identifiant unique pour chaque parcelle (`ID_PARCELL`).
* Calculer la surface de chaque parcelle en hectares.
* Assigner une séquence de cultures de manière aléatoire.
* Ajouter les colonnes restantes requises par MAELIA avec des valeurs fixes ou nulles.
* Sauvegarder le GeoDataFrame final au format Shapefile.

---
## 3. Fichiers en Entrée et en Sortie

### 3.1. Fichier en Entrée
* **Shapefile des Îlots :** `includes_sassemeV1/modeleAgricole/ilots/dansZone/ilots.shp`

### 3.2. Fichier en Sortie
* **Shapefile des Parcelles :** `includes_sassemeV1/modeleAgricole/ilots/dansZone/parcelles.shp`
---

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

In [2]:
# --- 1. CHEMINS ---
base_dir = Path.cwd().parent.resolve()

# Fichier en entrée
input_ilots_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "ilots" / "dansZone" / "ilots.shp"

# Fichier en sortie
output_parcelles_path = base_dir / "includes_sassemeV1" / "modeleAgricole" / "ilots" / "dansZone" / "parcelles.shp"

In [3]:
try:
    gdf_ilots = gpd.read_file(input_ilots_path)
    print(f"✅ Fichier 'ilots.shp' chargé avec succès ({len(gdf_ilots)} îlots).")
except Exception as e:
    print(f"🚨 ERREUR lors du chargement du fichier : {e}")

✅ Fichier 'ilots.shp' chargé avec succès (749 îlots).


In [4]:
# On travaille sur une copie pour garder l'original intact
gdf_parcelles = gdf_ilots.copy()

# 3a. Créer ID_PARCELL
# On convertit ID_ILOT en texte pour la concaténation
gdf_parcelles['ID_PARCELL'] = gdf_ilots['ID_ILOT'].astype(str) + '_001'
print("-> Colonne 'ID_PARCELL' créée.")

-> Colonne 'ID_PARCELL' créée.


In [6]:
# 3b. Calculer la SURFACE en hectares
# L'attribut .area calcule la surface en m², on divise par 10000
gdf_parcelles['SURFACE'] = gdf_parcelles.geometry.area / 10000
print("-> Colonne 'SURFACE' (en ha) calculée.")

-> Colonne 'SURFACE' (en ha) calculée.


In [7]:
# 3c. Assigner une SEQUENCE aléatoire
sequences_possibles = [
    "arachide_mil_jachere",
    "mil_arachide_jachere_arachide"
]
gdf_parcelles['SEQUENCE'] = np.random.choice(sequences_possibles, size=len(gdf_parcelles))
print("-> Colonne 'SEQUENCE' assignée aléatoirement.")

-> Colonne 'SEQUENCE' assignée aléatoirement.


In [8]:
# 3d. Ajouter les colonnes avec valeurs fixes
gdf_parcelles['POURCENTAG'] = 1.0
gdf_parcelles['INDEX_DEP'] = 'NA'
gdf_parcelles['CULT_REF'] = ''
gdf_parcelles['EXPREST'] = 'NA'
print("-> Colonnes à valeurs fixes ajoutées.")

-> Colonnes à valeurs fixes ajoutées.


In [9]:
colonnes_finales = [
    'ID_PARCELL', 'ID_ILOT', 'ID_EXPL', 'SEQUENCE', 'POURCENTAG',
    'INDEX_DEP', 'CULT_REF', 'SURFACE', 'EXPREST', 'geometry'
]
gdf_parcelles_final = gdf_parcelles[colonnes_finales]
print("-> Sélection et ordonnancement des colonnes finales terminés.")

-> Sélection et ordonnancement des colonnes finales terminés.


In [10]:
# --- 5. SAUVEGARDE ---
output_parcelles_path.parent.mkdir(parents=True, exist_ok=True)
gdf_parcelles_final.to_file(output_parcelles_path, driver='ESRI Shapefile', encoding='utf-8')
print(f"\n✅ Fichier 'parcelles.shp' sauvegardé dans :\n   {output_parcelles_path}")


✅ Fichier 'parcelles.shp' sauvegardé dans :
   C:\Users\Cheikhou\Desktop\Ferlo_Sine\maelia-data-diohine-v1\includes_sassemeV1\modeleAgricole\ilots\dansZone\parcelles.shp


In [11]:
print(f"\nDimensions finales : {gdf_parcelles_final.shape[0]} parcelles, {gdf_parcelles_final.shape[1]} colonnes.")
display(gdf_parcelles_final.head())


Dimensions finales : 749 parcelles, 10 colonnes.


,ID_PARCELL,ID_ILOT,ID_EXPL,SEQUENCE,POURCENTAG,INDEX_DEP,CULT_REF,SURFACE,EXPREST,geometry
0,1_001,1,SSM1-0001,mil_arachide_jachere_arachide,1.0,NA,,0.318122,NA,"MULTIPOLYGON (((337434.305 1603598.415, 337435..."
1,2_001,2,SSM1-0001,arachide_mil_jachere,1.0,NA,,0.586762,NA,"POLYGON ((337438.043 1603638.866, 337438.614 1..."
2,3_001,3,SSM1-0001,arachide_mil_jachere,1.0,NA,,0.104552,NA,"MULTIPOLYGON (((336159.144 1603477.128, 336160..."
3,4_001,4,SSM1-0001,mil_arachide_jachere_arachide,1.0,NA,,0.497774,NA,"POLYGON ((336165.254 1603521.577, 336167.671 1..."
4,5_001,5,SSM1-0001,arachide_mil_jachere,1.0,NA,,0.250301,NA,"MULTIPOLYGON (((336335.234 1602955.393, 336334..."
